# Session 06: Practice Quiz

**Notebook 2 of 2: Self-check** · about 35 minutes

For each question:

1. Write down your answer **before** opening "Show answer", or before running the code cell.
2. Run the cell where there is one.
3. Compare, and make sure you understand the *reasoning* in the answer, not only the result.

This quiz is formative: it's for you, and it isn't graded.

In [ ]:
from typing import Literal

from pydantic import BaseModel, Field, ValidationError

print("ready")

## Q1 · Predict: required or optional?

```python
class Note(BaseModel):
    text: str
    ticket_id: str | None
    author: str | None = None
    pinned: bool = False
```

Which of these succeed?

1. `Note.model_validate({"text": "Disk replaced", "ticket_id": None})`
2. `Note.model_validate({"text": "Disk replaced"})`
3. `Note.model_validate({"text": "Disk replaced", "ticket_id": "INC-1", "pinned": "yes"})`

In [ ]:
class Note(BaseModel):
    text: str
    ticket_id: str | None
    author: str | None = None
    pinned: bool = False


for data in [
    {"text": "Disk replaced", "ticket_id": None},
    {"text": "Disk replaced"},
    {"text": "Disk replaced", "ticket_id": "INC-1", "pinned": "yes"},
]:
    try:
        print("OK  ", Note.model_validate(data))
    except ValidationError as error:
        print("FAIL", [(e["loc"], e["type"]) for e in error.errors()])

<details>
<summary>Show answer</summary>

1. **Succeeds.** `ticket_id` is present, and `None` is allowed.
2. **Fails** with `ticket_id: missing`. `str | None` without a default is *required*; it just accepts `null`.
3. **Succeeds**, with `pinned=True`. In lax mode, booleans accept common text forms such as `"yes"`, `"true"`, `"1"` and `"on"`, which is handy for environment variables and query strings.

Only a **default** makes a field optional.

</details>

## Q2 · Predict: how many errors?

```python
class Ticket(BaseModel):
    id: str = Field(pattern=r"^INC-\d+$")
    priority: Literal["P1", "P2", "P3", "P4"]
    age_hours: int = Field(ge=0)
    tags: list[str]

Ticket.model_validate({"id": "INC-12", "priority": "p1", "age_hours": "-4", "tags": ["dns", None]})
```

How many errors does the `ValidationError` contain, and at which locations?

In [ ]:
class Ticket(BaseModel):
    id: str = Field(pattern=r"^INC-\d+$")
    priority: Literal["P1", "P2", "P3", "P4"]
    age_hours: int = Field(ge=0)
    tags: list[str]


try:
    Ticket.model_validate(
        {"id": "INC-12", "priority": "p1", "age_hours": "-4", "tags": ["dns", None]}
    )
except ValidationError as error:
    print(error.error_count(), [".".join(map(str, e["loc"])) for e in error.errors()])

<details>
<summary>Show answer</summary>

**Three**: `priority`, `age_hours` and `tags.1`.

- `"p1"` isn't one of the literal values. Literals are case-sensitive.
- `"-4"` is converted to `-4` (lax mode), and then fails the `ge=0` constraint. Conversion and constraints are separate stages.
- `None` isn't a string, and the location `tags.1` points at the exact list element.
- `id` is fine.

All problems are reported together, which is what makes Pydantic errors useful for a data owner.

</details>

## Q3 · Engineering judgment: lax or strict?

For each field, would you use the default lax mode or `strict=True`? Why?

1. `timeout_seconds: int` in settings read from environment variables.
2. `age_hours: int` in ticket records exported by another team's system as JSON.
3. `page: int` from a URL query string such as `?page=2` (Session 07).
4. `amount_cents: int` in a payment message from a partner.

<details>
<summary>Show answer</summary>

1. **Lax.** Environment variables are always text, so `"30"` must become `30`.
2. **Strict.** JSON has real numbers. `"12"` or `true` in a numeric field means the other system has a bug, and silently converting it hides that bug.
3. **Lax.** Query strings are text, too; FastAPI relies on this conversion.
4. **Strict**, and consider a dedicated type. Money is exactly where silent conversion is unacceptable (`12.5` → error, `"1200"` → error, `True` → error).

The rule: **text-based sources are lax; typed contracts are strict.**

</details>

## Q4 · Debugging: the validator that erased data

```python
class Asset(BaseModel):
    id: str
    team: str

    @field_validator("team")
    @classmethod
    def clean_team(cls, value: str) -> str:
        value = value.strip().lower()

print(Asset.model_validate({"id": "db-03", "team": " Database "}))
```

The output is `id='db-03' team=None`. Why, and why didn't Pydantic complain about `None` in a `str` field?

<details>
<summary>Show answer</summary>

The validator doesn't `return value`, so it returns `None`, and **whatever an after-validator returns is what's stored**. Pydantic doesn't re-validate a validator's return value against the field type; you're trusted to return the right thing.

Fix: `return value.strip().lower()`. A unit test that checks the cleaned value (`asset.team == "database"`) catches this immediately; that's why the workbook's acceptance tests check normalisation explicitly.

</details>

## Q5 · Implementation: a model validator

Write `MaintenanceWindow` with `starts_at: datetime`, `ends_at: datetime` and `team: str`. Reject windows whose end isn't after their start, and windows longer than 8 hours. The cell tests three cases.

In [ ]:
from datetime import datetime, timedelta

# Your MaintenanceWindow model goes here.


start = datetime(2026, 10, 24, 1, 0)
for hours in [2, -1, 9]:
    try:
        window = MaintenanceWindow(
            starts_at=start, ends_at=start + timedelta(hours=hours), team="database"
        )
        print(f"{hours:>2} h: OK")
    except ValidationError as error:
        print(f"{hours:>2} h:", error.errors()[0]["msg"])
    except NameError:
        print("define MaintenanceWindow first")
        break

<details>
<summary>Show answer</summary>

```python
from pydantic import model_validator

class MaintenanceWindow(BaseModel):
    starts_at: datetime
    ends_at: datetime
    team: str

    @model_validator(mode="after")
    def check_duration(self) -> "MaintenanceWindow":
        duration = self.ends_at - self.starts_at
        if duration <= timedelta(0):
            raise ValueError("ends_at must be after starts_at")
        if duration > timedelta(hours=8):
            raise ValueError("a maintenance window may last at most 8 hours")
        return self
```

Expected output:

```text
 2 h: OK
-1 h: Value error, ends_at must be after starts_at
 9 h: Value error, a maintenance window may last at most 8 hours
```

It's a **model** validator because the rule involves two fields. Pydantic prefixes messages from `ValueError` with "Value error, ". And don't forget `return self`.

</details>

## Q6 · Predict: serialisation

In [ ]:
from datetime import UTC, datetime


class Breach(BaseModel):
    id: str
    team: str
    age_hours: int
    note: str | None = None


class Report(BaseModel):
    generated_at: datetime
    breaches: tuple[Breach, ...]


report = Report(
    generated_at=datetime(2026, 10, 21, 2, 0, tzinfo=UTC),
    breaches=(Breach(id="INC-3007", team="it-support", age_hours=200),),
)
print(report.model_dump()["generated_at"].__class__.__name__)
print(report.model_dump_json(exclude_none=True))
print(Report.model_validate_json(report.model_dump_json()) == report)

<details>
<summary>Show answer</summary>

```text
datetime
{"generated_at":"2026-10-21T02:00:00Z","breaches":[{"id":"INC-3007","team":"it-support","age_hours":200}]}
True
```

- `model_dump()` keeps Python objects (the `datetime` stays a `datetime`); `model_dump(mode="json")` and `model_dump_json()` convert to JSON types. UTC is written as `Z`.
- The tuple becomes a JSON array; `exclude_none=True` removes the `note: null` entries at every level.
- The round trip is exact, because validation turns the array back into a tuple of `Breach` models.

</details>

## Q7 · Debugging: two settings failures

A colleague's tool crashes at start-up in two different ways, depending on the machine:

```text
Machine A:
pydantic_core._pydantic_core.ValidationError: 1 validation error for Settings
timeout_seconds
  Input should be greater than 0 [type=greater_than, input_value='0', input_type=str]

Machine B:
pydantic_settings.exceptions.SettingsError: error parsing value for field "sla_hours" from source "EnvSettingsSource"
```

1. What is wrong on each machine?
2. The code has `except ValidationError:` around `Settings()`. Why does machine B still crash with a traceback?

<details>
<summary>Show answer</summary>

1. **A:** `TIMEOUT_SECONDS` (with whatever prefix) is set to `0`, which parses as an integer and then breaks the `> 0` constraint. **B:** the `SLA_HOURS` variable isn't valid JSON, so it can't even be parsed into a dict.
2. Parsing failures in pydantic-settings raise **`SettingsError`**, which is not a `ValidationError`. It happens before validation starts. Catch both and turn each into a clear message, as the Project 02 `cli.py` does:

```python
try:
    settings = Settings()
except SettingsError as error:
    ...
except ValidationError as error:
    ...
```

</details>

## Q8 · Implementation: map errors for a data owner

Write `summarise(error)`, returning a list like `["priority: Input should be ...", "age_hours: ..."]`, and `"record: ..."` when the location is empty. Then use it on the cell's input.

In [ ]:
def summarise(error: ValidationError) -> list[str]: ...  # your implementation


for data in [{"id": "INC-1", "priority": "P9", "age_hours": "x", "tags": []}, "not a ticket"]:
    try:
        Ticket.model_validate(data)
    except ValidationError as error:
        print(summarise(error))

<details>
<summary>Show answer</summary>

```python
def summarise(error: ValidationError) -> list[str]:
    return [
        f"{'.'.join(str(part) for part in issue['loc']) or 'record'}: {issue['msg']}"
        for issue in error.errors(include_url=False)
    ]
```

Expected:

```text
["priority: Input should be 'P1', 'P2', 'P3' or 'P4'", 'age_hours: Input should be a valid integer, unable to parse string as an integer']
['record: Input should be a valid dictionary or instance of Ticket']
```

Note `str(part)` for list indexes in `loc`, and `include_url=False`, which keeps documentation links out of the message.

</details>

## Q9 · Comparison

1. In Java you might annotate a DTO with `@NotNull @Min(0) Integer ageHours` and call `Validator.validate(dto)` after Jackson has deserialised the JSON. What does Pydantic do differently in that pipeline?
2. A TypeScript team uses zod. What's the closest equivalent of `z.infer<typeof Schema>` in Pydantic?

<details>
<summary>Show answer</summary>

1. Java splits **parsing** (Jackson creates the object, possibly failing with a type error) and **validation** (a separate pass that collects constraint violations), and an invalid object can exist between the two. Pydantic does conversion, constraints and custom rules in **one** step, `model_validate` or `model_validate_json`: either you get a valid instance, or no instance at all and one error listing everything.
2. The model class **is** the type. A Pydantic model is both the runtime schema (like a zod schema) and the static type your editor and type checker use (like `z.infer`), so there's no separate step to derive one from the other.

</details>

## Q10 · Engineering judgment: where does validation belong?

The incident service will accept tickets three ways: export files (today), a FastAPI endpoint (Session 07), and LLM-extracted tickets from free text (Session 09). A colleague proposes one shared `Ticket` model for all three. Another wants a separate model per source. What would you do?

<details>
<summary>Show answer</summary>

A good answer keeps **one core `Ticket` model** for the rules that are true everywhere (ID format, priorities, non-negative age, normalised team), because those are business rules, not source rules.

Source-specific differences belong in **input models** that convert into it:

- an API request model with `extra="forbid"` (unknown fields are probably client bugs) and no server-assigned fields;
- an LLM-output model that tolerates what models typically get wrong, such as `"P1 - critical"`, and normalises it before building a `Ticket`;
- the export path, which uses `Ticket` directly.

One model for everything tends to accumulate "if it came from X" branches. Many unrelated models duplicate the business rules. The middle path is a shared core plus thin adapters at each boundary.

</details>

## How did you do?

- **Q1–Q3** cover fields, constraints and strictness: Parts 1–2 of `01-concepts.ipynb`.
- **Q4, Q5** cover validators: Part 4.
- **Q6** covers serialisation: Part 5.
- **Q7** covers settings: Part 6.
- **Q8** covers error mapping: Part 3. It's the core of the workbook's `Rejection.from_validation_error`.
- **Q9, Q10** cover comparison and design.

Next: the workbook in [`../workbook/assignment.md`](../workbook/assignment.md), which starts **Project 02**.